# KYC Risk Scorer — End-to-End SageMaker Walkthrough

This notebook runs the full ML pipeline **step by step on AWS SageMaker** so you can observe real
logs and outputs at every stage before automating it via `pipeline.py`.

| Step | What runs | Where |
|------|-----------|-------|
| 1 | Load real KYC dataset, fix column names, upload to S3 | Local |
| 2 | **Preprocess** — encode ordinals, 80/20 stratified split | SageMaker Processing (SKLearn) |
| 3 | **Train** — XGBoost `multi:softprob`, 3 risk classes | SageMaker Training (XGBoost) |
| 4 | **Evaluate** — AUC, accuracy, per-class F1, confusion matrix | SageMaker Processing (SKLearn) |
| 5 | **Quality gate** — AUC ≥ threshold check | Local |
| 6 | **Register** — push to SageMaker Model Registry | SageMaker |
| 7 | **Deploy** — real-time endpoint + smoke test | SageMaker |
| 8 | **Cleanup** — delete endpoint to stop charges | SageMaker |

**Prerequisites:**
```
pip install sagemaker boto3 xgboost pandas scikit-learn matplotlib seaborn openpyxl
```
You also need a SageMaker execution role ARN and an S3 bucket — fill both in the **Configuration** cell below.

In [1]:
!pip install sagemaker boto3 xgboost pandas scikit-learn matplotlib seaborn openpyxl

In [1]:
import sys
print(sys.executable)
import sagemaker
print(sagemaker.__file__)
print(sagemaker.__version__)

/opt/conda/bin/python


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
import sys, sagemaker
print("file:", sagemaker.__file__)
print("path:", list(sagemaker.__path__))
print()
print("sys.path entries:")
for p in sys.path:
    print(" ", p)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


file: /opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py
path: ['/opt/conda/lib/python3.12/site-packages/sagemaker']

sys.path entries:
  /opt/conda/lib/python312.zip
  /opt/conda/lib/python3.12
  /opt/conda/lib/python3.12/lib-dynload
  
  /opt/conda/lib/python3.12/site-packages


In [3]:
# ── Standard library ──────────────────────────────────────────────────────────
import io
import json
import os
import tarfile
import time

# ── Third-party ───────────────────────────────────────────────────────────────
import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sagemaker
import xgboost as xgb
from IPython.display import display

# Quick version check so you know what you're running
print(f'sagemaker : {sagemaker.__version__}')
print(f'boto3     : {boto3.__version__}')
print(f'xgboost   : {xgb.__version__}')
print(f'pandas    : {pd.__version__}')
print(f'numpy     : {np.__version__}')

sagemaker : 2.257.3
boto3     : 1.43.40
xgboost   : 2.1.4
pandas    : 3.0.3
numpy     : 2.5.1


sagemaker : 2.257.3
boto3     : 1.43.40
xgboost   : 2.1.4
pandas    : 3.0.3
numpy     : 2.5.1


## ⚙️ 0. Configuration

**Fill in `ROLE` and `BUCKET` before running any cell below.**
Everything else has sensible defaults that match the `pipeline.py` settings.

In [8]:
# ── Required: update these two ────────────────────────────────────────────────
ROLE   = 'arn:aws:iam::926529379586:role/kyc-agent-roles-sagemaker-execution-role'
BUCKET = 'kyc-agent-mlops-926529379586-us-east-1'               # your S3 bucket (must already exist)

# ── Optional overrides ────────────────────────────────────────────────────────
REGION          = 'us-east-1'      # AWS region
AUC_THRESHOLD   = 0.85             # quality gate: model must beat this to register
ENDPOINT_NAME   = 'kyc-risk-scorer'
INSTANCE_PROC   = 'ml.m5.large'    # processing jobs (preprocess + evaluate)
INSTANCE_TRAIN  = 'ml.m5.xlarge'   # training job
INSTANCE_INFER  = 'ml.m5.large'    # real-time endpoint

# S3 URI prefixes (all artefacts land under the same bucket)
S3_RAW        = f's3://{BUCKET}/kyc-risk-raw/training'
S3_PROCESSED  = f's3://{BUCKET}/kyc-risk-processed'
S3_MODEL      = f's3://{BUCKET}/kyc-risk-model'
S3_EVALUATION = f's3://{BUCKET}/kyc-risk-evaluation'

# AWS SDK clients
boto_sess  = boto3.Session(region_name=REGION)
session    = sagemaker.Session(boto_session=boto_sess)
s3_client  = boto3.client('s3',         region_name=REGION)
sm_client  = boto3.client('sagemaker',  region_name=REGION)
rt_client  = boto3.client('sagemaker-runtime', region_name=REGION)

print(f'Role    : {ROLE}')
print(f'Bucket  : {BUCKET}')
print(f'Region  : {REGION}')

Role    : arn:aws:iam::926529379586:role/kyc-agent-roles-sagemaker-execution-role
Bucket  : kyc-agent-mlops-926529379586-us-east-1
Region  : us-east-1


## 📂 Step 1: Load Dataset & Upload to S3

We use the real KYC cases from `dataset/kyc_train.csv` and `dataset/kyc_test.csv`.

**Column name mismatch to fix before uploading:**

| Dataset column | `preprocess.py` expects |
|----------------|-------------------------|
| `doc_verification_status` | `doc_status` |
| `adverse_media_hit_count` | `adverse_media_hits` |

We merge both splits into one file and upload it — `preprocess.py` will do its own 80/20 stratified re-split on SageMaker.

In [9]:
# Path is relative to this notebook; scripts/ and dataset/ live alongside it
DATASET_DIR = ''

df_train_raw = pd.read_csv(os.path.join(DATASET_DIR, 'kyc_train.csv'))
df_test_raw  = pd.read_csv(os.path.join(DATASET_DIR, 'kyc_test.csv'))

print(f'Train file : {len(df_train_raw)} rows')
print(f'Test file  : {len(df_test_raw)} rows')

# ── Rename to match preprocess.py's expected column names ─────────────────────
RENAME = {
    'doc_verification_status': 'doc_status',
    'adverse_media_hit_count' : 'adverse_media_hits',
}
df_train_raw = df_train_raw.rename(columns=RENAME)
df_test_raw  = df_test_raw.rename(columns=RENAME)

# ── Merge into one CSV (preprocess.py re-splits on SageMaker) ─────────────────
df_combined = pd.concat([df_train_raw, df_test_raw], ignore_index=True)
print(f'\nCombined   : {len(df_combined)} rows')
print(f'Columns    : {list(df_combined.columns)}')

# ── Label distribution ────────────────────────────────────────────────────────
print('\nLabel distribution:')
vc = df_combined['final_risk_label'].value_counts()
for label, count in vc.items():
    bar = '█' * count
    print(f'  {label:<8}: {count:>3}  {bar}')

# ── Preview raw data ──────────────────────────────────────────────────────────
print('\nSample rows (raw, before encoding):')
display(df_combined.head(8))

Train file : 120 rows
Test file  : 30 rows

Combined   : 150 rows
Columns    : ['pep_match_score', 'pep_match_type', 'sanctions_hit', 'doc_status', 'doc_authenticity_score', 'adverse_media_hits', 'adverse_media_severity', 'country_risk_tier', 'final_risk_label']

Label distribution:
  low     :  90  ██████████████████████████████████████████████████████████████████████████████████████████
  medium  :  37  █████████████████████████████████████
  high    :  23  ███████████████████████

Sample rows (raw, before encoding):


,pep_match_score,pep_match_type,sanctions_hit,doc_status,doc_authenticity_score,adverse_media_hits,adverse_media_severity,country_risk_tier,final_risk_label
0,57,fuzzy,1,verified,83,4,4,high,high
1,17,none,0,verified,81,0,0,low,low
2,17,none,0,unreadable,50,0,0,medium,medium
3,1,none,0,verified,97,0,0,low,low
4,18,none,0,failed,4,1,4,medium,medium
5,10,none,0,verified,91,2,1,low,low
6,5,none,0,verified,90,0,0,low,low
7,4,none,0,verified,91,2,3,low,medium


In [11]:
# ── Upload combined CSV to S3 ─────────────────────────────────────────────────
buf = io.StringIO()
df_combined.to_csv(buf, index=False)

s3_key        = 'kyc-risk-raw/training/kyc_combined.csv'
input_s3_uri  = f's3://{BUCKET}/{s3_key}'

s3_client.put_object(
    Bucket=BUCKET,
    Key=s3_key,
    Body=buf.getvalue().encode('utf-8'),
)

print(f'Uploaded {len(df_combined)} rows')
print(f'S3 URI : {input_s3_uri}')

Uploaded 150 rows
S3 URI : s3://kyc-agent-mlops-926529379586-us-east-1/kyc-risk-raw/training/kyc_combined.csv


## 🔄 Step 2: Preprocess on SageMaker

`SKLearnProcessor` runs `scripts/preprocess.py` in a managed container on `ml.m5.large`.

**Encoding applied by the script:**

| Feature | Encoding |
|---------|----------|
| `pep_match_type` | none→0, fuzzy→1, exact→2 |
| `doc_status` | failed→0, unreadable→1, verified→2 |
| `adverse_media_severity` | none→0, low→1, medium→2, high→3, critical→4 |
| `country_risk_tier` | low→1, medium→2, high→3, very_high→4, critical→5 |
| `final_risk_label` | low→0, medium→1, high/rejected→2 |
| `pep_match_score` | clipped to [0, 100] |
| `doc_authenticity_score` | clipped to [0, 100] |
| `adverse_media_hits` | clipped to [0, 50] |

**Output:** two headerless CSVs with **label as column 0** (XGBoost convention) — `train.csv` (80%) and `test.csv` (20%).

Job logs stream inline below. Typical runtime: **3–5 minutes** on first run (container pull + job).

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_proc = SKLearnProcessor(
    framework_version='1.2-1',
    instance_type=INSTANCE_PROC,
    instance_count=1,
    role=ROLE,
    sagemaker_session=session,
    base_job_name='kyc-preprocess',
)

print('Submitting preprocessing job …')
print(f'  Input  : {input_s3_uri}')
print(f'  Output : {S3_PROCESSED}/train  and  {S3_PROCESSED}/test')
print()

sklearn_proc.run(
    # Path is relative to the notebook's working directory (sagemaker/)
    code='scripts/preprocess.py',
    inputs=[
        ProcessingInput(
            source=input_s3_uri,
            destination='/opt/ml/processing/input',
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='train',
            source='/opt/ml/processing/output/train',
            destination=f'{S3_PROCESSED}/train',
        ),
        ProcessingOutput(
            output_name='test',
            source='/opt/ml/processing/output/test',
            destination=f'{S3_PROCESSED}/test',
        ),
    ],
    arguments=['--test-size', '0.2', '--random-seed', '42'],
    wait=True,   # block until job finishes, streaming logs inline
    logs=True,
)

preprocess_job_name = sklearn_proc.latest_job.job_name
print(f'\n✓ Preprocessing complete')
print(f'  Job name : {preprocess_job_name}')

In [ ]:
# ── Download the preprocessed splits from S3 and inspect them ─────────────────
#
# preprocess.py writes headerless CSVs with label in column 0 — that is the
# XGBoost built-in CSV loader convention. We add column names here for display.

FEATURE_NAMES = [
    'pep_match_score', 'sanctions_hit', 'doc_authenticity_score',
    'adverse_media_hits', 'adverse_media_severity',
    'pep_match_type', 'doc_status', 'country_risk_tier',
]
LABEL_DECODE = {0: 'low', 1: 'medium', 2: 'high'}


def download_processed_split(s3_prefix: str, filename: str) -> pd.DataFrame:
    """Download a processed CSV from S3 and attach column headers."""
    key = s3_prefix.replace(f's3://{BUCKET}/', '') + f'/{filename}'
    obj = s3_client.get_object(Bucket=BUCKET, Key=key)
    df  = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)
    df.columns = ['label'] + FEATURE_NAMES
    return df


df_proc_train = download_processed_split(f'{S3_PROCESSED}/train', 'train.csv')
df_proc_test  = download_processed_split(f'{S3_PROCESSED}/test',  'test.csv')

print(f'Train split : {len(df_proc_train)} rows   |   '
      f'Test split : {len(df_proc_test)} rows')

# ── Label distribution after encoding ─────────────────────────────────────────
print('\nTrain label distribution (encoded → decoded):')
for code, count in df_proc_train['label'].value_counts().sort_index().items():
    print(f'  {code} → {LABEL_DECODE[code]:<8}: {count} rows')

# ── Feature statistics ────────────────────────────────────────────────────────
print('\nTrain feature summary (all values are now numeric):')
display(df_proc_train[FEATURE_NAMES].describe().round(2))

# ── Sample rows showing the label-first, headerless format ───────────────────
print('\nFirst 5 rows of train.csv (col-0 = label, then 8 features):')
display(df_proc_train.head(5))

## 🏋️ Step 3: Train XGBoost on SageMaker

`XGBoost.fit()` launches a managed training job using `scripts/train.py` as the entry point.

**Objective:** `multi:softprob` — predicts a probability vector `[P(low), P(medium), P(high)]` per case.

**Hyperparameters (matching `pipeline.py`):**

| Parameter | Value | Role |
|-----------|-------|------|
| `num-round` | 150 | max boosting rounds |
| `eta` | 0.1 | learning rate — lower = more conservative |
| `max-depth` | 5 | maximum tree depth |
| `subsample` | 0.8 | row sampling per tree (prevents overfitting) |
| `colsample-bytree` | 0.8 | feature sampling per tree |
| `min-child-weight` | 5 | min samples in a leaf (regularisation) |
| `early-stopping-rounds` | 15 | stop if validation loss doesn't improve for 15 rounds |

The **test split** is passed as the `validation` channel for early stopping — it is **not used** for gradient updates.

`mlogloss` and `merror` are logged every 10 rounds. Training typically finishes in **4–8 minutes**.

In [ ]:
from sagemaker.xgboost.estimator import XGBoost
from sagemaker.inputs import TrainingInput

xgb_estimator = XGBoost(
    entry_point='train.py',
    source_dir='scripts',          # directory containing train.py, relative to this notebook
    framework_version='1.7-1',
    instance_type=INSTANCE_TRAIN,
    instance_count=1,
    role=ROLE,
    sagemaker_session=session,
    output_path=S3_MODEL,
    base_job_name='kyc-train',
    hyperparameters={
        'max-depth'            : 5,
        'eta'                  : 0.1,
        'num-round'            : 150,
        'subsample'            : 0.8,
        'colsample-bytree'     : 0.8,
        'min-child-weight'     : 5,
        'early-stopping-rounds': 15,
    },
)

print('Submitting training job …')
print(f'  Train channel      : {S3_PROCESSED}/train')
print(f'  Validation channel : {S3_PROCESSED}/test  (used for early stopping only)')
print(f'  Output             : {S3_MODEL}')
print()

xgb_estimator.fit(
    inputs={
        'train': TrainingInput(
            s3_data=f'{S3_PROCESSED}/train',
            content_type='text/csv',
        ),
        # The test split acts as validation for early stopping during training.
        # It is held out completely and re-used in the evaluation step.
        'validation': TrainingInput(
            s3_data=f'{S3_PROCESSED}/test',
            content_type='text/csv',
        ),
    },
    wait=True,
    logs='All',   # stream all container logs inline so you can watch loss per round
)

training_job_name  = xgb_estimator.latest_training_job.name
model_artifact_uri = xgb_estimator.model_data

print(f'\n✓ Training complete')
print(f'  Job name       : {training_job_name}')
print(f'  Model artifact : {model_artifact_uri}')

In [ ]:
# ── Download model.tar.gz and inspect metadata written by train.py ─────────────
#
# train.py writes model_meta.json inside the tarball alongside xgboost-model.
# It records: feature names, label map, best boosting round, and train AUC.

EXTRACT_DIR = '/tmp/kyc_model'
os.makedirs(EXTRACT_DIR, exist_ok=True)

model_s3_key = model_artifact_uri.replace(f's3://{BUCKET}/', '')
obj          = s3_client.get_object(Bucket=BUCKET, Key=model_s3_key)
model_bytes  = obj['Body'].read()

with tarfile.open(fileobj=io.BytesIO(model_bytes), mode='r:gz') as tar:
    tar.extractall(EXTRACT_DIR)

print(f'Extracted model artifact to {EXTRACT_DIR}/')
print(f'Contents: {os.listdir(EXTRACT_DIR)}')

with open(os.path.join(EXTRACT_DIR, 'model_meta.json')) as f:
    meta = json.load(f)

print('\nmodel_meta.json:')
print(json.dumps(meta, indent=2))

print(f'\nBest boosting round  : {meta["best_iteration"]}')
print(f'Train AUC (macro OVR): {meta["train_auc_macro_ovr"]}')
print(f'Feature order        : {meta["feature_names"]}')

## 📊 Step 4: Evaluate on SageMaker

A second `SKLearnProcessor` job runs `scripts/evaluate.py`, which:

1. Extracts `model.tar.gz` and loads the XGBoost booster
2. Predicts class probabilities on the **test split** (never seen by the optimiser)
3. Computes macro-OVR AUC, accuracy, per-class precision/recall/F1
4. Writes `evaluation.json` to S3 (the pipeline's `ConditionStep` reads AUC from this file)

After the job completes we download `evaluation.json` and plot the confusion matrix and feature importance.

> `evaluate.py` installs `xgboost` at runtime inside the SKLearn container — that's why you see a
> `pip install` line in its logs.

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

eval_proc = SKLearnProcessor(
    framework_version='1.2-1',
    instance_type=INSTANCE_PROC,
    instance_count=1,
    role=ROLE,
    sagemaker_session=session,
    base_job_name='kyc-evaluate',
)

print('Submitting evaluation job …')
print(f'  Model artifact : {model_artifact_uri}')
print(f'  Test data      : {S3_PROCESSED}/test')
print(f'  Output         : {S3_EVALUATION}')
print()

eval_proc.run(
    code='scripts/evaluate.py',
    inputs=[
        # model.tar.gz is downloaded to the container as-is; evaluate.py extracts it
        ProcessingInput(
            source=model_artifact_uri,
            destination='/opt/ml/processing/input/model',
        ),
        # Held-out test split — the model has never seen these labels
        ProcessingInput(
            source=f'{S3_PROCESSED}/test',
            destination='/opt/ml/processing/input/test',
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name='evaluation',
            source='/opt/ml/processing/evaluation',
            destination=S3_EVALUATION,
        )
    ],
    wait=True,
    logs=True,
)

eval_job_name = eval_proc.latest_job.job_name
print(f'\n✓ Evaluation complete')
print(f'  Job name : {eval_job_name}')
print(f'  Report   : {S3_EVALUATION}/evaluation.json')

In [ ]:
# ── Download evaluation.json and display metrics ───────────────────────────────

eval_s3_key = S3_EVALUATION.replace(f's3://{BUCKET}/', '') + '/evaluation.json'
obj         = s3_client.get_object(Bucket=BUCKET, Key=eval_s3_key)
evaluation  = json.loads(obj['Body'].read())

auc_val = evaluation['metrics']['auc_macro_ovr']['value']
acc_val = evaluation['metrics']['accuracy']['value']

print('=' * 55)
print('  KYC Risk Scorer — Evaluation Report (Test Set)')
print('=' * 55)
print(f'\n  AUC  (macro OVR) : {auc_val:.4f}')
print(f'  Accuracy         : {acc_val:.4f}')
print()

# ── Per-class table ────────────────────────────────────────────────────────────
print(f'  {"Class":<10}  {"Precision":>9}  {"Recall":>8}  {"F1":>6}  {"Support":>8}')
print('  ' + '-' * 50)
for cls, m in evaluation['per_class'].items():
    print(f'  {cls:<10}  {m["precision"]:>9.3f}  {m["recall"]:>8.3f}  '
          f'{m["f1"]:>6.3f}  {m["support"]:>8}')

# ── Full JSON for reference ────────────────────────────────────────────────────
print('\nFull evaluation.json:')
print(json.dumps(evaluation, indent=2))

In [ ]:
# ── Confusion matrix heatmap ───────────────────────────────────────────────────
#
# Rows  = true labels (what the case actually was)
# Cols  = predicted labels (what the model said)
# The diagonal is correct predictions; off-diagonal = errors.

cm_values = np.array(evaluation['confusion_matrix']['values'])
cm_labels = evaluation['confusion_matrix']['labels']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_values,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=cm_labels,
    yticklabels=cm_labels,
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Count'},
)
ax.set_ylabel('True label', fontsize=12)
ax.set_xlabel('Predicted label', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# ── Per-class accuracy summary ────────────────────────────────────────────────
print('Per-class accuracy:')
for i, label in enumerate(cm_labels):
    total   = int(cm_values[i].sum())
    correct = int(cm_values[i, i])
    pct     = correct / total * 100 if total else 0
    print(f'  {label:<8}: {correct}/{total}  ({pct:.1f}%)')

In [ ]:
# ── Feature importance — load the booster we extracted in Step 3 ───────────────
#
# XGBoost names features f0, f1, … f7 internally (by position).
# We map them back to the real column names using FEATURE_NAMES.
#
# Importance type = 'gain' : total reduction in loss brought by splits on this feature.
# Higher gain → feature is more useful for the model's decisions.

booster = xgb.Booster()
booster.load_model(os.path.join(EXTRACT_DIR, 'xgboost-model'))

raw_importance = booster.get_score(importance_type='gain')

# Map f0..f7 → human-readable feature names
named_importance = {
    FEATURE_NAMES[int(k[1:])]: v
    for k, v in raw_importance.items()
}

# Sort descending by gain
sorted_feats = sorted(named_importance, key=named_importance.get, reverse=True)
sorted_vals  = [named_importance[f] for f in sorted_feats]
max_val      = max(sorted_vals)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(sorted_feats[::-1], sorted_vals[::-1], color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=9)
ax.set_xlabel('Importance (Gain)', fontsize=11)
ax.set_title('Feature Importance — XGBoost Gain', fontsize=13, pad=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# ── ASCII bar chart for quick scanning ────────────────────────────────────────
print('Feature importance ranking (gain):')
for feat, val in zip(sorted_feats, sorted_vals):
    bar_len = int(val / max_val * 30)
    print(f'  {feat:<28}  {"█" * bar_len:<30}  {val:.1f}')

## ✅ Step 5: Quality Gate — AUC Check

The automated pipeline uses a `ConditionStep` with `AUC ≥ 0.85` as the gate.
Below we reproduce that check so you can see the decision logic before registering manually.

In [ ]:
# ── Quality gate: does the model meet the AUC threshold? ─────────────────────

auc    = evaluation['metrics']['auc_macro_ovr']['value']
passes = auc >= AUC_THRESHOLD

GREEN, RED, RESET = '\033[92m', '\033[91m', '\033[0m'
colour = GREEN if passes else RED
symbol = '✓ PASS' if passes else '✗ FAIL'

print(f'AUC (macro OVR) : {auc:.4f}')
print(f'Threshold       : {AUC_THRESHOLD}')
print(f'Decision        : {colour}{symbol}{RESET}')

if passes:
    print('\nModel qualifies for registration — proceed to Step 6.')
else:
    print('\nAUC is below threshold. The pipeline FailStep would trigger.')
    print('Options: add more training data, tune hyperparameters, or lower AUC_THRESHOLD.')

## 📦 Step 6: Register Model in SageMaker Model Registry

Registers the trained model as a new **version** in the `kyc-risk-scorer` package group.
Status is `PendingManualApproval` — a compliance officer must approve it in the
SageMaker console before it can be deployed.

The evaluation metrics are attached to the version so reviewers can inspect AUC directly
from the Model Registry UI without navigating to S3.

In [ ]:
from sagemaker import image_uris
from sagemaker.model_metrics import MetricsSource, ModelMetrics

MODEL_PACKAGE_GROUP = 'kyc-risk-scorer'

# ── Ensure the package group exists ───────────────────────────────────────────
try:
    sm_client.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print(f'Model package group "{MODEL_PACKAGE_GROUP}" already exists.')
except sm_client.exceptions.ClientError:
    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription='KYC risk scorer — XGBoost 3-class risk tier',
    )
    print(f'Created model package group "{MODEL_PACKAGE_GROUP}".')

# ── Retrieve the XGBoost inference container image for this region ─────────────
xgb_image = image_uris.retrieve(
    framework='xgboost',
    region=REGION,
    version='1.7-1',
    instance_type=INSTANCE_INFER,
    image_scope='inference',
)
print(f'\nXGBoost inference image : {xgb_image}')

# ── Register the model with evaluation metrics attached ───────────────────────
response = sm_client.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageDescription=(
        f'AUC={auc:.4f} | acc={acc_val:.4f} | '
        f'trained by {training_job_name}'
    ),
    InferenceSpecification={
        'Containers': [
            {
                'Image'        : xgb_image,
                'ModelDataUrl' : model_artifact_uri,
            }
        ],
        'SupportedContentTypes'              : ['text/csv'],
        'SupportedResponseMIMETypes'         : ['application/json'],
        'SupportedTransformInstanceTypes'    : ['ml.m5.large'],
        'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge'],
    },
    ModelApprovalStatus='PendingManualApproval',
    ModelMetrics={
        'ModelStatistics': {
            'ContentType' : 'application/json',
            'S3Uri'       : f'{S3_EVALUATION}/evaluation.json',
        }
    },
)

pkg_arn = response['ModelPackageArn']
print(f'\n✓ Model registered')
print(f'  ARN    : {pkg_arn}')
print(f'  Status : PendingManualApproval')
print(f'\nApprove in: SageMaker Console → Models → Model Registry → {MODEL_PACKAGE_GROUP}')
print('Then run Step 7 to deploy.')

## 🚀 Step 7: Deploy to Real-Time Endpoint

Deploys the model to a persistent real-time endpoint with data capture enabled.

> **Cost reminder:** endpoints are billed per-hour. Delete it in Step 8 when done.

**Feature order the endpoint expects (comma-separated CSV string):**
```
pep_match_score, sanctions_hit, doc_authenticity_score,
adverse_media_hits, adverse_media_severity,
pep_match_type, doc_status, country_risk_tier
```
This matches `FEATURE_NAMES` above and what `risk_scoring_tool.py` sends.

**Response format:** comma-separated probabilities `P(low),P(medium),P(high)` — the class
with the highest probability is the predicted risk tier.

Deployment typically takes **5–8 minutes**.

In [ ]:
from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker import image_uris

xgb_image = image_uris.retrieve(
    framework='xgboost',
    region=REGION,
    version='1.7-1',
    instance_type=INSTANCE_INFER,
    image_scope='inference',
)

model = Model(
    image_uri=xgb_image,
    model_data=model_artifact_uri,
    role=ROLE,
    sagemaker_session=session,
    name=f'kyc-risk-scorer-{int(time.time())}',
)

# Data capture records every request + response to S3 for later analysis
data_capture_cfg = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{BUCKET}/kyc-risk-capture/{ENDPOINT_NAME}',
    capture_options=['INPUT', 'OUTPUT'],
)

print(f'Deploying to endpoint "{ENDPOINT_NAME}" on {INSTANCE_INFER} …')
print('(This takes 5–8 minutes)')

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_INFER,
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_cfg,
    wait=True,
)

print(f'\n✓ Endpoint "{ENDPOINT_NAME}" is InService')
print(f'Set  RISK_SCORER_ENDPOINT_NAME={ENDPOINT_NAME}  in your Lambda environment.')

In [ ]:
# ── Smoke test: invoke the endpoint with representative cases ──────────────────
#
# Payload format: pep_match_score, sanctions_hit, doc_authenticity_score,
#                 adverse_media_hits, adverse_media_severity,
#                 pep_match_type, doc_status, country_risk_tier

LABEL_MAP = {0: 'low', 1: 'medium', 2: 'high'}

smoke_cases = [
    ('Clean case  (expect: low)',   '0,0,95,0,0,0,2,1'),
    ('Fuzzy PEP   (expect: med)',  '35,0,65,2,1,1,1,3'),
    ('High risk   (expect: high)', '85,1,20,8,3,2,0,5'),
]

print('Smoke test results')
print('=' * 65)

for description, payload in smoke_cases:
    resp  = rt_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='text/csv',
        Body=payload,
    )
    raw    = resp['Body'].read().decode().strip()
    probas = [float(x) for x in raw.split(',')]
    pred   = LABEL_MAP[int(np.argmax(probas))]

    print(f'\n  {description}')
    print(f'  Input     : {payload}')
    print(f'  P(low)    : {probas[0]:.4f}')
    print(f'  P(medium) : {probas[1]:.4f}')
    print(f'  P(high)   : {probas[2]:.4f}')
    print(f'  → Predicted risk tier : {pred.upper()}')

print()
print('=' * 65)
print('Endpoint is healthy. Data capture is writing to:')
print(f's3://{BUCKET}/kyc-risk-capture/{ENDPOINT_NAME}/')

## 🧹 Step 8: Cleanup

Delete the real-time endpoint to stop per-hour charges.
All S3 artefacts (model, processed data, evaluation report, data capture) remain.

To delete everything: navigate to S3 and remove the `kyc-risk-*` prefixes under your bucket.

In [ ]:
# Uncomment the line below and run this cell to delete the endpoint.
# predictor.delete_endpoint()
# print(f'Endpoint "{ENDPOINT_NAME}" deleted.')

# Or delete it directly via boto3:
# sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)

print(f'When ready, delete endpoint "{ENDPOINT_NAME}" to stop charges.')
print('Uncomment predictor.delete_endpoint() above and re-run this cell.')